# 01 — Embedding Models (Text → Vectors)
### LangChain Foundations · Part 2 of 2

Chat models answer questions. Embedding models do something completely different: they turn text
into a list of numbers (a **vector**) that captures its *meaning*. This is the foundation everything
in `08_Vector_Store/` and `09_Retriever/` is built on — so it's worth understanding properly now,
not just importing and moving on.

## The core idea: meaning as coordinates

Imagine every sentence you've ever read gets a GPS coordinate — not on a map, but in "meaning-space."
Sentences with similar meaning land close together. "My internet is down" and "I have no wifi" end up
near each other, even though they don't share a single word. "My internet is down" and "What's the
weather today?" end up far apart.

An embedding model's whole job is: **text in → coordinate out**. That coordinate typically has
hundreds or thousands of dimensions (not just 2 or 3), but the idea is identical to GPS — distance
between points means similarity of meaning.

## Which of your providers actually offer this?

Not every chat provider offers embeddings — this trips people up constantly:

| Provider | Embeddings? | Notes |
|---|---|---|
| **OpenAI** | ✅ Yes | `text-embedding-3-small` / `text-embedding-3-large` — cheap, reliable, industry default |
| **Google Gemini** | ✅ Yes, and free | `models/text-embedding-004` (stable) or the newer `gemini-embedding-001` |
| **Groq** | ❌ No | Groq is an *inference speed* company for chat models only — no embedding endpoint exists |
| **Anthropic** | ❌ No | Claude has never shipped a native embeddings API — Anthropic officially points people to Voyage AI instead |
| **Hugging Face** | ✅ Yes, fully free, runs locally | Thousands of open embedding models — no API key needed at all |

So for this notebook, we'll use **OpenAI**, **Gemini**, and **Hugging Face** — and Hugging Face is
genuinely the best one to *learn* on, because it costs nothing and needs zero setup.

## Setup

In [ ]:
# %pip install -q langchain-openai langchain-google-genai langchain-huggingface sentence-transformers numpy


In [1]:
import os
import numpy as np
from dotenv import load_dotenv

load_dotenv()
print("Environment ready.")


Environment ready.


---
## 1. `embed_query` vs. `embed_documents` — the one API detail that matters

Every LangChain embedding model exposes exactly two methods:

| Method | Input | Use it for |
|---|---|---|
| `.embed_query(text)` | one string | Embedding a user's search query, right before you search |
| `.embed_documents([text1, text2, ...])` | a list of strings | Embedding a whole batch of documents/chunks at once (more efficient) |

They can return *slightly* different vectors for the same text on some providers (some models are
tuned differently for "queries" vs. "things being searched"), so don't mix them up — always embed
your document collection with `embed_documents` and your search text with `embed_query`.

---
## 2. Hugging Face — free, local, no API key (start here)

`sentence-transformers/all-MiniLM-L6-v2` is the classic starting embedding model: 22M parameters,
runs on a laptop CPU in milliseconds, 384-dimensional vectors, no signup, no token, no internet call
after the first download.

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

hf_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector = hf_embeddings.embed_query("My internet has been down since this morning.")
print(f"Vector length: {len(vector)}")
print(f"First 8 numbers: {vector[:8]}")


d:\Gen Ai\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Gen Ai\myvenv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Deepak Kumar\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, se

Vector length: 384
First 8 numbers: [-0.005433671176433563, -0.0008587235352024436, 0.04944908991456032, -0.008638249710202217, -0.018938234075903893, -0.029770035296678543, -0.058352358639240265, 0.04904749616980553]


---
## 3. OpenAI — `text-embedding-3-small`

Cheap enough that cost is rarely the deciding factor ($0.02 per 1M tokens). Supports a neat trick:
`dimensions` lets you shrink the vector (e.g. from 1536 down to 256) for faster search and smaller
storage, at a small accuracy cost — useful once you're storing millions of vectors.

In [3]:
from langchain_openai import OpenAIEmbeddings

openai_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector = openai_embeddings.embed_query("My internet has been down since this morning.")
print(f"Vector length: {len(vector)}")

# Shrunk version — same idea, smaller footprint:
compact_embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=256)
compact_vector = compact_embeddings.embed_query("My internet has been down since this morning.")
print(f"Compact vector length: {len(compact_vector)}")


Vector length: 1536
Compact vector length: 256


---
## 4. Google Gemini — free embeddings

`models/text-embedding-004` is the stable, well-established option. Google's newer
`gemini-embedding-001` tops several public leaderboards as of 2026 if you want the strongest
quality and don't mind it being newer/less battle-tested.

In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Update model parameter string to the active Gemini standard
gemini_embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# Re-run vector test
vector = gemini_embeddings.embed_query("My internet has been down since this morning.")
print(f"Vector length: {len(vector)}")


Vector length: 3072


---
## 5. Measuring "closeness": cosine similarity from scratch

This is the actual math behind semantic search — and it's short enough to write yourself once so
it stops being a black box. Cosine similarity measures the *angle* between two vectors: `1.0` means
identical meaning, `0` means unrelated, negative means opposite.

In [6]:
def cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
    a, b = np.array(vec_a), np.array(vec_b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# Sanity check: these two should score high (similar meaning, different words)
v1 = hf_embeddings.embed_query("My internet connection is not working")
v2 = hf_embeddings.embed_query("I have no wifi at home")
v3 = hf_embeddings.embed_query("What time does the pizza place close?")

print("internet vs wifi   :", round(cosine_similarity(v1, v2), 3))
print("internet vs pizza  :", round(cosine_similarity(v1, v3), 3))


internet vs wifi   : 0.512
internet vs pizza  : 0.065


---
## Real-world capstone: a semantic support-ticket router

This is the exact problem embeddings solve in production: you have a pile of past support tickets
that were already resolved, each tagged with a category. A new ticket comes in — instead of a human
reading it and guessing the category, embed it and find the **closest existing ticket** by meaning.
No keyword matching, no regex, no exact-word overlap required.

We'll do this with Hugging Face's free local model, so this cell runs for anyone, zero API keys needed.

In [7]:
# A small "knowledge base" of already-categorized tickets.
past_tickets = [
    {"text": "My internet has been out since 8am, no lights on the router", "category": "Connectivity"},
    {"text": "I was charged twice for my monthly plan",                     "category": "Billing"},
    {"text": "Can't log into my account, password reset email never arrives","category": "Account Access"},
    {"text": "Wifi keeps dropping every few minutes during video calls",     "category": "Connectivity"},
    {"text": "I want a refund for a charge I don't recognize",              "category": "Billing"},
    {"text": "Two-factor code never gets sent to my phone",                 "category": "Account Access"},
]

# Embed the whole knowledge base once (batch call — this is what embed_documents is for).
ticket_texts = [t["text"] for t in past_tickets]
ticket_vectors = hf_embeddings.embed_documents(ticket_texts)


def route_ticket(new_ticket: str) -> dict:
    query_vector = hf_embeddings.embed_query(new_ticket)

    scored = [
        (cosine_similarity(query_vector, vec), ticket)
        for vec, ticket in zip(ticket_vectors, past_tickets)
    ]
    best_score, best_match = max(scored, key=lambda x: x[0])

    return {
        "predicted_category": best_match["category"],
        "confidence": round(best_score, 3),
        "most_similar_past_ticket": best_match["text"],
    }


new_ticket = "The connection to my router keeps cutting out randomly"
result = route_ticket(new_ticket)
print(f"New ticket: {new_ticket!r}\n")
for k, v in result.items():
    print(f"  {k}: {v}")


New ticket: 'The connection to my router keeps cutting out randomly'

  predicted_category: Connectivity
  confidence: 0.539
  most_similar_past_ticket: Wifi keeps dropping every few minutes during video calls


Notice what just happened: the new ticket says **"connection to my router keeps cutting out"** —
it shares almost no exact words with the matched ticket **"Wifi keeps dropping every few minutes"** —
yet it correctly routed to *Connectivity*. That's semantic search working exactly as intended.

### Try it yourself
1. Add 3–4 more categories and past tickets of your own (e.g. "Feature Request", "Bug Report").
2. Swap `hf_embeddings` for `openai_embeddings` in `route_ticket` and compare — do the confidence scores change?
3. Feed in a genuinely ambiguous ticket (could fit two categories) and see which one wins, and by how much — this is where you'd add a "confidence threshold below which a human reviews it" rule in a real system.
4. `08_Vector_Store/` picks up exactly here — instead of a Python list + a for-loop, you'll store these vectors in FAISS/Chroma so this scales to millions of tickets instead of six.

---
## Quick reference: dimensions & rough cost

| Model | Dimensions | Cost | Runs where |
|---|---|---|---|
| `sentence-transformers/all-MiniLM-L6-v2` (HF) | 384 | Free | Locally, on your machine |
| `text-embedding-3-small` (OpenAI) | 1536 (shrinkable) | ~$0.02 / 1M tokens | OpenAI's servers |
| `text-embedding-3-large` (OpenAI) | 3072 (shrinkable) | ~$0.13 / 1M tokens | OpenAI's servers |
| `models/text-embedding-004` (Gemini) | 768 | Free | Google's servers |
| `models/gemini-embedding-001` (Gemini) | up to 3072 | Free (rate-limited) | Google's servers |

**Rule of thumb:** start with the free Hugging Face local model while you're learning and prototyping
— switch to OpenAI or Gemini's embeddings only once you need the extra quality (or need to embed at a
scale your laptop can't handle) in a real deployment.

---
**Next:** `02_Prompts/` — now that you can talk to models and turn text into searchable vectors, the next step is controlling *exactly* what you send them.
